In [0]:
from pyspark.sql.functions import to_date

# Product table data
product_data = [
    (1, "LC Phone"),
    (2, "LC T-Shirt"),
    (3, "LC Keychain")
]
product_columns = ["product_id", "product_name"]
product_df = spark.createDataFrame(product_data, product_columns)

# Sales table data
sales_data = [
    (1, "2019-01-25", "2019-02-28", 100),
    (2, "2018-12-01", "2020-01-01", 10),
    (3, "2019-12-01", "2020-01-31", 1)
]
sales_columns = ["product_id", "period_start", "period_end", "average_daily_sales"]
sales_df = spark.createDataFrame(sales_data, sales_columns)

# Convert period_start and period_end to date type
sales_df = sales_df.withColumn("period_start", to_date("period_start")) \
                   .withColumn("period_end", to_date("period_end"))

product_df.createOrReplaceTempView("product")
sales_df.createOrReplaceTempView("sales")

display(product_df)
display(sales_df)

In [0]:
spark.sql("""
    WITH YearCTE AS (
        SELECT '2018' AS report_year, DATE '2018-01-01' AS year_start, DATE '2018-12-31' AS year_end
        UNION ALL
        SELECT '2019' AS report_year, DATE '2019-01-01' AS year_start, DATE '2019-12-31' AS year_end
        UNION ALL
        SELECT '2020' AS report_year, DATE '2020-01-01' AS year_start, DATE '2020-12-31' AS year_end
    )
    SELECT 
        s.product_id,
        y.report_year,
        (DATEDIFF(LEAST(s.period_end, y.year_end), GREATEST(s.period_start, y.year_start)) + 1) * s.average_daily_sales AS total_amount
    FROM sales s 
    JOIN YearCTE y
        ON s.period_start <= y.year_end AND s.period_end >= y.year_start
""").display()


In [0]:
WITH YearCTE AS (
    SELECT '2018' AS report_year, '2018-01-01' AS year_start, '2018-12-31' AS year_end
    UNION ALL
    SELECT '2019' AS report_year, '2019-01-01' AS year_start, '2019-12-31' AS year_end
    UNION ALL
    SELECT '2020' AS report_year, '2020-01-01' AS year_start, '2020-12-31' AS year_end
)

SELECT 
    s.product_id,
    p.product_name,
    y.report_year,
    (DATEDIFF(LEAST(s.period_end, y.year_end), GREATEST(s.period_start, y.year_start)) + 1) * s.average_daily_sales AS total_amount
FROM Sales s
JOIN Product p ON s.product_id = p.product_id
JOIN YearCTE y ON s.period_start <= y.year_end AND s.period_end >= y.year_start
ORDER BY s.product_id, y.report_year;